In [0]:
# Databricks notebook source
# ==========================================
# PHASE 1: BRONZE LAYER INGESTION
# ==========================================

from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp

# Initialize Spark session
spark = SparkSession.builder.appName("SupplyChainBronzeIngestion").getOrCreate()

# Using your exact volume file path
file_path = (
    "dbfs:/Volumes/workspace/default/retail_purchase/DataCoSupplyChainDataset.csv"
)

# Read raw CSV with header, inferred schema, and Latin-1 encoding for special characters
df_raw = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("encoding", "ISO-8859-1")
    .load(file_path)
)

# Clean column names by replacing spaces and special characters with underscores
cleaned_columns = [
    c.strip()
    .replace(" ", "_")
    .replace("(", "")
    .replace(")", "")
    .replace(".", "")
    for c in df_raw.columns
]
df_bronze = df_raw.toDF(*cleaned_columns)

# Add ingestion metadata timestamp
df_bronze = df_bronze.withColumn("ingestion_timestamp", current_timestamp())

# Write raw dataframe to a managed Delta Lake table
bronze_table_name = "bronze_supply_chain_orders"
(
    df_bronze.write.format("delta")
    .mode("overwrite")
    .saveAsTable(bronze_table_name)
)

print(
    f"Successfully ingested {df_bronze.count()} rows into Bronze Delta table: {bronze_table_name}"
)

# Validate the ingestion
display(spark.sql(f"SELECT * FROM {bronze_table_name} LIMIT 5"))

In [0]:
# Databricks notebook source
# ==========================================
# PHASE 2: SILVER LAYER TRANSFORMATION (UPDATED)
# ==========================================

from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.appName(
    "SupplyChainSilverTransformation"
).getOrCreate()

# Read from Bronze table
df_bronze = spark.read.table("bronze_supply_chain_orders")

# Let's dynamically find the order date column or use a safe fallback
date_col = next(
    (c for c in df_bronze.columns if "order_date" in c.lower()), None
)

# Clean data, handle data types, and derive analytical features
df_silver = (
    df_bronze
    # Cast important numeric fields
    .withColumn(
        "Order_Item_Quantity", col("Order_Item_Quantity").cast("int")
    )
    .withColumn("Sales", col("Sales").cast("float"))
    .withColumn("Order_Profit_Per_Order", col("Order_Profit_Per_Order").cast("float"))
    .withColumn("Late_delivery_risk", col("Late_delivery_risk").cast("int"))
    # Handle missing or negative quantities/prices
    .filter(
        (col("Order_Item_Quantity") > 0) & (col("Sales") >= 0)
    )
    # Select and rename core operational columns
    .select(
        col("Order_Id").alias("order_id"),
        col("Customer_Id").alias("customer_id"),
        col("Customer_City").alias("customer_city"),
        col("Customer_State").alias("customer_state"),
        col("Customer_Country").alias("customer_country"),
        col("Market").alias("market"),
        col("Order_Region").alias("order_region"),
        col("Category_Name").alias("product_category"),
        col("Product_Name").alias("product_name"),
        col("Product_Price").cast("float").alias("product_price"),
        col("Order_Item_Quantity").alias("quantity"),
        col("Sales").alias("sales"),
        col("Order_Profit_Per_Order").alias("profit"),
        col("Shipping_Mode").alias("shipping_mode"),
        col(
            "Days_for_shipping_real"
        ).cast("int").alias("days_for_shipping_real"),
        col(
            "Days_for_shipment_scheduled"
        ).cast("int").alias("days_for_shipment_scheduled"),
        col("Late_delivery_risk").alias("late_delivery_risk"),
        col("ingestion_timestamp"),
    )
)

# Write to Silver Delta table
silver_table_name = "silver_supply_chain_orders"
(
    df_silver.write.format("delta")
    .mode("overwrite")
    .saveAsTable(silver_table_name)
)

print(
    f"Successfully processed and stored {df_silver.count()} rows into Silver Delta table: {silver_table_name}"
)

# Preview clean Silver data
display(spark.sql(f"SELECT * FROM {silver_table_name} LIMIT 5"))

In [0]:
# Databricks notebook source
# ==========================================
# PHASE 3: GOLD LAYER AGGREGATIONS
# ==========================================

from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, col, count, sum

spark = SparkSession.builder.appName(
    "SupplyChainGoldAndML"
).getOrCreate()

# Read from Silver table
df_silver = spark.read.table("silver_supply_chain_orders")

# Create Gold aggregated data mart for BI (Late delivery rate by region & shipping mode)
df_gold_metrics = (
    df_silver.groupBy("order_region", "shipping_mode", "product_category")
    .agg(
        count("order_id").alias("total_orders"),
        sum("late_delivery_risk").alias("total_late_orders"),
        avg("days_for_shipping_real").alias("avg_actual_shipping_days"),
        sum("sales").alias("total_sales"),
        sum("profit").alias("total_profit"),
    )
    .withColumn(
        "late_delivery_rate_pct",
        (col("total_late_orders") / col("total_orders")) * 100,
    )
)

# Write to Gold Delta table
gold_table_name = "gold_supply_chain_metrics"
(
    df_gold_metrics.write.format("delta")
    .mode("overwrite")
    .saveAsTable(gold_table_name)
)

print(
    f"Successfully built Gold analytical mart table: {gold_table_name}"
)
display(spark.sql(f"SELECT * FROM {gold_table_name} LIMIT 5"))

# ==========================================
# PHASE 4: MACHINE LEARNING (SCIKIT-LEARN)
# ==========================================

import mlflow
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.model_selection import train_test_split

# Pull data into pandas for Scikit-learn training (sampling a subset for fast community edition performance)
df_pd = df_silver.select(
    "product_price",
    "quantity",
    "days_for_shipment_scheduled",
    "late_delivery_risk",
).toPandas()

# Drop any nulls
df_pd = df_pd.dropna()

# Define features (X) and target (y)
X = df_pd[
    [
        "product_price",
        "quantity",
        "days_for_shipment_scheduled",
    ]
]
y = df_pd["late_delivery_risk"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Enable MLflow autologging to track metrics & models automatically in Databricks
mlflow.sklearn.autolog()

with mlflow.start_run(run_name="supply_chain_risk_classifier"):
    # Train Random Forest Classifier
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)

    # Predictions & Evaluation
    predictions = model.predict(X_test)
    acc = accuracy_score(y_test, predictions)
    auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])

    print(f"Model Accuracy: {acc:.4f}")
    print(f"Model ROC-AUC Score: {auc:.4f}")

In [0]:
# Convert Gold delta table to pandas and save locally as CSV
df_gold_pd = spark.read.table("gold_supply_chain_metrics").toPandas()
df_gold_pd.to_csv("/tmp/gold_supply_chain_metrics.csv", index=False)

In [0]:
# Write the complete gold table directly into your workspace volume path using Spark
spark.read.table("gold_supply_chain_metrics").coalesce(1).write.format(
    "csv"
).option("header", "true").mode("overwrite").save(
    "/Volumes/workspace/default/retail_purchase/gold_export_csv"
)

print(
    "Successfully exported table to volume path: /Volumes/workspace/default/retail_purchase/gold_export_csv"
)